# NARPS potential gain group analysis

**Question.** In the 54-participant equal-indifference group, is the gain-effect estimate positive in vmPFC? The directional group hypothesis is a positive response with increasing potential gain.

The analysis uses the supplied AO86 participant contrast estimates as fixed inputs. These are not t/z statistics and, because the upstream gain modulator was normalized, the coefficients are not per-currency-unit slopes. The full pipeline is in `scripts/group_analysis.py`; this notebook runs it and presents the outputs.


## Analysis plan

- Check cohort/map identity, image geometry, and finite values; exclude only for a prespecified integrity failure.
- Fit a one-sample group t model across participant effects in the intersection-like coverage mask (nonzero in at least 90% of maps).
- Test the directional alternative with 5,000 participant sign flips and whole-brain maximum-t familywise correction (α = .05).
- Localize vmPFC anatomically in supplied MNI space (|x| ≤ 15, y = 25–60, z = −20–10 mm). Its ROI summary is descriptive; it does not replace required whole-brain inference.


In [ ]:
# Run the authoritative analysis and create the maps, tables, and figures.
from pathlib import Path
import runpy
ROOT = Path.cwd()
runpy.run_path(str(ROOT / "scripts" / "group_analysis.py"))


## Results

The code above prints the key analysis values. This cell reloads the saved summary and cluster table so the reported result stays linked to the generated artifacts.


In [ ]:
import json
import pandas as pd
summary = json.loads((ROOT / "results" / "summary.json").read_text())
cluster_table = pd.read_csv(ROOT / "results" / "fwe_clusters.csv")
for key in ["n_participants", "n_excluded", "n_tested_voxels", "n_permutations",
            "permutation_maxT_95pct", "n_voxels_fwe05_signflip", "peak_t",
            "peak_p_fwe_signflip", "vmPFC_mean_effect", "vmPFC_mean_effect_95ci",
            "vmPFC_min_p_fwe_signflip"]:
    print(f"{key}: {summary[key]}")
cluster_table


## Figures and interpretation

The first figure centers on the vmPFC locator; corrected positive voxels are absent there. The second shows the largest corrected posterior cluster. The participant-level vmPFC plot gives the between-person distribution and interval. The corrected clusters lie in posterior cortex, so they do not support the vmPFC hypothesis. A null vmPFC result does not establish zero response; upstream normalized regressors, no participant variance maps, and the group model limit what can be inferred. See `report.md` for the concise full interpretation and references.


In [ ]:
from IPython.display import display, Image
display(Image(filename=str(ROOT / "figures" / "whole_brain_fwe_and_vmpfc.png")))
display(Image(filename=str(ROOT / "figures" / "posterior_fwe_cluster.png")))
display(Image(filename=str(ROOT / "figures" / "vmpfc_participant_effects.png")))


## Reproducibility notes

The exact permutation seed and each input file checksum are saved in `results/summary.json`. Maps are reported on the supplied MNI152NLin2009cAsym grid. The 90% coverage rule, symmetric sign-flip assumption, finite Monte Carlo resolution, absent varcope maps, and descriptive coordinate-box vmPFC definition are documented in `report.md`.
